# 15 — VRP branch, step 1: pricing and variance-convention audit

**Why this comes first** (discussion 018 §2.1). Module 12's own output compares realized and implied variance with opposite signs at two horizons, e.g. BTC median $\log(RV/W)$ is **+0.33** to settlement but **−0.11** over the next 30 minutes. Before any forecast-vs-price test, we must know whether that is a real horizon structure or a convention problem.

The notebook checks, in order:

| Part | Question | Pre-declared pass condition |
|---|---|---|
| A | Do quoted near-ATM prices reprice with Black-76 using the exchange IV and **our** time to expiry? | median \|model − mid\| ≤ 0.25 × half-spread |
| B | Is the exchange IV computed on the same time basis as ours? | fitted offset \|δ\| ≤ 2 minutes |
| C | Does the ATM IV feature (`option_iv_atm_forward`) match the IV implied by the ATM straddle? | median \|log ratio\| ≤ 2%, no TTE trend beyond ±3% |
| D | Which definitional ingredient (flat vs clock allocation, settlement weighting, sample) creates the terminal/30m sign flip? | descriptive: every flip attributed |
| E | Is 5-minute index RV inflated by microstructure noise? | median RV(5m)/RV(30m) within [0.8, 1.25] |

Nothing is fitted except the training-only clock profile used to reproduce Module 12's allocation. No result feeds a model. This is development data; the branch's confirmation window starts after the 30th Module 12 confirmation session (018 §2.4).

In [ ]:
from pathlib import Path
import os, sys, warnings
import numpy as np
import pandas as pd
import duckdb
from dotenv import load_dotenv

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 250)
pd.set_option("display.max_rows", 1000)

load_dotenv()
DATA_ROOT = Path(os.environ["DERIBIT_DATA_ROOT"]).expanduser().resolve()
DB_PATH = DATA_ROOT / "database" / "research.duckdb"
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src" / "data_engineering").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
for _m in [m for m in sys.modules if m.split(".")[0] == "data_engineering"]:
    del sys.modules[_m]

from data_engineering.features import regime_inputs as ri
from data_engineering.modeling import vol_targets as vt
from data_engineering.modeling import vol_har as vh
from data_engineering.modeling import option_pricing as op
from data_engineering.modeling import vrp_audit as va

ASSETS = ["AVAX", "BTC", "ETH", "HYPE", "SOL", "XRP"]
START = os.getenv("VRP_START", "2026-09-22")
END = os.getenv("VRP_END", "2026-11-06")          # VRP holdout starts after the Module 12 binding readout
CLOCK_TRAIN_END = os.getenv("VRP_CLOCK_TRAIN_END", "2026-10-07")
ATM_K = 0.05                                      # |log moneyness| for contract-level checks
SETTLE_MIN = 30.0
OUT = PROJECT_ROOT / "notebooks" / "Volatility_Risk_Premium"
OUT.mkdir(parents=True, exist_ok=True)
print(f"option_pricing {op.OPTION_PRICING_VERSION} | vrp_audit {va.VRP_AUDIT_VERSION} | sessions {START} -> {END}")

## 1. Load state-level data and near-ATM contract quotes

Contract quotes come from `normalized_option_chain`, restricted to each state's **primary option family** (the family behind `option_iv_atm_forward`), $|k|\le 5\%$, outside the 30-minute settlement window.

In [ ]:
con = duckdb.connect(str(DB_PATH), read_only=True)


def columns(name):
    return set(con.execute(f"DESCRIBE {name}").df()["column_name"])


ts_cols = columns("time_series_feature_state")
need_ts = {"state_id", "state_time", "underlying", "return_5m", "futures_index_price", "is_weekend",
           "in_settlement_window", "option_atm_quote_ok", "option_iv_atm_forward"}
missing = sorted(need_ts - ts_cols)
if missing:
    raise RuntimeError(f"time_series_feature_state lacks {missing}")
last_bar = pd.Timestamp(f"{END} 08:00", tz="UTC")
df = con.execute(
    f"SELECT {', '.join(sorted(need_ts))} FROM time_series_feature_state "
    f"WHERE state_time <= TIMESTAMPTZ '{last_bar.isoformat()}' ORDER BY underlying, state_time"
).df()
df["state_time"] = pd.to_datetime(df["state_time"], utc=True)
df = df[df["underlying"].isin(ASSETS)].reset_index(drop=True)
df["session_date"] = ri.session_key(df["state_time"])
df = df[((df["session_date"] >= START) & (df["session_date"] <= END)) | (df["state_time"] == last_bar)].reset_index(drop=True)
SESSIONS = sorted(s for s in df["session_date"].unique() if START <= s <= END)
CLOCK_TRAIN = {s for s in SESSIONS if s <= CLOCK_TRAIN_END}
print(f"state rows {len(df):,} | sessions {len(SESSIONS)} ({SESSIONS[0]} -> {SESSIONS[-1]}) | clock-train sessions {len(CLOCK_TRAIN)}")

chain_cols = columns("normalized_option_chain")
prim_cols = columns("primary_option_surface_feature_state")
need_chain = {"state_id", "state_time", "underlying", "option_family_id", "instrument_name", "option_type", "strike",
              "best_bid_price", "best_ask_price", "bid_iv", "ask_iv", "mark_iv", "option_forward_price",
              "time_to_expiry_years_365", "minutes_to_expiry", "log_moneyness", "instrument_type"}
missing = sorted(need_chain - chain_cols)
if missing:
    raise RuntimeError(f"normalized_option_chain lacks {missing}")
if not {"state_id", "option_family_id"} <= prim_cols:
    raise RuntimeError("primary_option_surface_feature_state needs state_id and option_family_id")

chain = con.execute(
    f"""
    WITH prim AS (SELECT DISTINCT state_id, option_family_id FROM primary_option_surface_feature_state)
    SELECT {', '.join('n.' + c for c in sorted(need_chain))}
    FROM normalized_option_chain n
    JOIN prim p ON p.state_id = n.state_id AND p.option_family_id = n.option_family_id
    WHERE ABS(n.log_moneyness) <= {ATM_K}
      AND n.minutes_to_expiry > {SETTLE_MIN}
      AND n.state_time <= TIMESTAMPTZ '{last_bar.isoformat()}'
    """
).df()
con.close()

chain["state_time"] = pd.to_datetime(chain["state_time"], utc=True)
chain = chain[chain["underlying"].isin(ASSETS)].copy()
chain["session_date"] = ri.session_key(chain["state_time"])
chain = chain[(chain["session_date"] >= START) & (chain["session_date"] <= END)].copy()
chain["is_call"] = chain["option_type"].astype(str).str.lower().str[0].eq("c")
chain["inverse"] = op.is_inverse(chain["instrument_type"])
for c in ["bid_iv", "ask_iv", "mark_iv"]:
    v = pd.to_numeric(chain[c], errors="coerce")
    chain[c] = v / 100.0 if np.nanmedian(v) > 3 else v
F = chain["option_forward_price"].to_numpy(float)
for side in ["best_bid_price", "best_ask_price"]:
    chain[side + "_usd"] = op.to_usd(pd.to_numeric(chain[side], errors="coerce").to_numpy(float), F, chain["inverse"])
two = (chain["best_bid_price_usd"] > 0) & (chain["best_ask_price_usd"] >= chain["best_bid_price_usd"])
chain["mid_usd"] = np.where(two, 0.5 * (chain["best_bid_price_usd"] + chain["best_ask_price_usd"]), np.nan)
chain["half_spread_usd"] = np.where(two, 0.5 * (chain["best_ask_price_usd"] - chain["best_bid_price_usd"]), np.nan)
chain["iv_mid_ex"] = np.where((chain["bid_iv"] > 0) & (chain["ask_iv"] >= chain["bid_iv"]),
                              0.5 * (chain["bid_iv"] + chain["ask_iv"]), np.nan)
T_db = pd.to_numeric(chain["time_to_expiry_years_365"], errors="coerce")
T_mte = pd.to_numeric(chain["minutes_to_expiry"], errors="coerce") / vt.MINUTES_PER_YEAR
chain["T"] = T_db.where(T_db > 0, T_mte)
chain["tte_bucket"] = va.tte_bucket(pd.to_numeric(chain["minutes_to_expiry"], errors="coerce").to_numpy(float)).astype(str)
for c in ["strike", "option_forward_price", "minutes_to_expiry"]:
    chain[c] = pd.to_numeric(chain[c], errors="coerce")
print(f"near-ATM contract rows {len(chain):,} | families:",
      chain.groupby("underlying")["option_family_id"].agg(lambda s: sorted(s.unique())).to_dict())

# Filter funnel: shows which input is missing if a later part comes out empty.
both_T = (T_db > 0) & (T_mte > 0)
funnel = pd.Series({
    "rows": len(chain),
    "two_sided_quote": int(two.sum()),
    "bid_iv > 0": int((chain["bid_iv"] > 0).sum()),
    "ask_iv > 0": int((chain["ask_iv"] > 0).sum()),
    "two_sided & bid/ask IV": int((two & (chain["bid_iv"] > 0) & (chain["ask_iv"] >= chain["bid_iv"])).sum()),
    "mark_iv > 0": int((chain["mark_iv"] > 0).sum()),
    "T from time_to_expiry_years_365": int((T_db > 0).sum()),
    "T from minutes_to_expiry": int((T_mte > 0).sum()),
    "forward > 0": int((chain["option_forward_price"] > 0).sum()),
    "calls": int(chain["is_call"].sum()),
})
print("Input funnel:")
display(funnel.to_frame("count"))
print("option_type values:", chain["option_type"].astype(str).value_counts().head(6).to_dict())
if both_T.any():
    print("max |T_db - T_minutes| (minutes):",
          float(np.nanmax(np.abs(T_db[both_T] - T_mte[both_T])) * vt.MINUTES_PER_YEAR))

## Part A — Black-76 repricing with the exchange IV and our time to expiry

For every two-sided near-ATM quote, each side is repriced from its own exchange IV: bid model $=\text{Black76}(F,K,T_{ours},\sigma^{ex}_{bid})$, ask likewise. (Averaging bid and ask IVs before pricing would add a convexity bias.) The error is expressed in half-spreads. A systematic error growing toward expiry indicates a time-basis or unit problem.

In [ ]:
# All two-sided quotes with a usable forward, strike and T (also used by Part C).
q2 = chain[(chain["half_spread_usd"] > 0) & (chain["T"] > 0) & (chain["option_forward_price"] > 0)
           & (chain["strike"] > 0)].copy()
a = q2[(q2["bid_iv"] > 0) & (q2["ask_iv"] >= q2["bid_iv"])].copy()
IV_SOURCE = "bid/ask IV"
if a.empty:
    # Fallback: the exchange mark IV against our IV of the quote mid (carries a small convexity bias).
    a = q2[q2["mark_iv"] > 0].copy()
    a["bid_iv"] = a["ask_iv"] = a["mark_iv"]
    IV_SOURCE = "mark IV (no bid/ask IV in the data)"
print(f"Part A/B rows: {len(a):,} using {IV_SOURCE}")
Fa, Ka, Ta, Ca = a["option_forward_price"], a["strike"], a["T"], a["is_call"]
a["model_bid_usd"] = op.black76_usd(Fa, Ka, Ta, a["bid_iv"], Ca)
a["model_ask_usd"] = op.black76_usd(Fa, Ka, Ta, a["ask_iv"], Ca)
if IV_SOURCE == "bid/ask IV":
    a["err_halfspreads"] = 0.5 * ((a["model_bid_usd"] - a["best_bid_price_usd"]) + (a["model_ask_usd"] - a["best_ask_price_usd"])) / a["half_spread_usd"]
else:
    a["err_halfspreads"] = (a["model_bid_usd"] - a["mid_usd"]) / a["half_spread_usd"]
a["abs_err_hs"] = a["err_halfspreads"].abs()

partA = (a.groupby(["underlying", "tte_bucket"], observed=True)
           .agg(n=("abs_err_hs", "size"), median_abs_err_hs=("abs_err_hs", "median"),
                median_signed_err_hs=("err_halfspreads", "median"),
                median_rel_half_spread=("half_spread_usd", lambda s: float(np.median(s / a.loc[s.index, "mid_usd"]))))
           .reset_index())
overall_A = a.groupby("underlying")["abs_err_hs"].median()
print("Repricing error (model - mid) in half-spreads:")
display(partA.round(3))
print("Overall median |error| (half-spreads):", overall_A.round(3).to_dict())

## Part B — Time basis of the exchange IV

Invert the quoted bid and ask with **our** $T$ and compare each with the exchange's IV for the same side. If the exchange uses $T+\delta$, then $\log(\sigma_{ours}/\sigma_{ex})=\tfrac12\log(1+\delta/T)$; the fitted $\delta$ is reported in minutes. Weights are inverse squared relative half-spreads.

**Vega filter (rerun 2).** The |k| <= 0.05 universe is many standard deviations out of the money near expiry: prices sit near the minimum tick and IV inversion is ill-conditioned (a tick moves IV by tens of percent while the repricing error stays a fraction of a half-spread). The gate fit uses only rows with standardized moneyness $z=|\ln(K/F)|/(\sigma_{ex}\sqrt T)\le 1$ (roughly 16-84 delta). A genuine offset implies the same $\delta_b=\bar T_b\,(e^{2m_b}-1)$ in every TTE bucket $b$ (median log ratio $m_b$); a $\delta_b$ that shrinks with $T$ is not a time-basis offset.

In [ ]:
if IV_SOURCE == "bid/ask IV":
    iv_b = op.implied_vol(a["best_bid_price_usd"], Fa, Ka, Ta, Ca)
    iv_a = op.implied_vol(a["best_ask_price_usd"], Fa, Ka, Ta, Ca)
    a["log_iv_ratio"] = 0.5 * (np.log(iv_b / a["bid_iv"]) + np.log(iv_a / a["ask_iv"]))
else:
    a["log_iv_ratio"] = np.log(op.implied_vol(a["mid_usd"], Fa, Ka, Ta, Ca) / a["mark_iv"])
w = 1.0 / np.maximum(a["half_spread_usd"] / a["mid_usd"], 1e-3) ** 2
iv_ex_mid = 0.5 * (a["bid_iv"] + a["ask_iv"])
a["z_moneyness"] = np.abs(np.log(a["strike"] / a["option_forward_price"])) / (iv_ex_mid * np.sqrt(a["T"]))
Z_MAX = 1.0
Z_LABEL = f"z<={Z_MAX:g}"
a["vega_ok"] = a["z_moneyness"] <= Z_MAX

partB = []
for asset, g in a.groupby("underlying"):
    for subset, gg in (("all", g), (Z_LABEL, g[g["vega_ok"]])):
        res = va.time_basis_offset(gg["log_iv_ratio"].to_numpy(), gg["T"].to_numpy(), w.loc[gg.index].to_numpy())
        res.update({"underlying": asset, "subset": subset,
                    "median_log_ratio": float(np.nanmedian(gg["log_iv_ratio"])) if len(gg) else np.nan})
        partB.append(res)
partB = pd.DataFrame(partB)
print("Exchange IV time basis (delta = minutes the exchange adds to our time to expiry); the gate uses", Z_LABEL)
display(partB.round(4))

def _tte_profile(df, subset):
    g = df.groupby(["underlying", "tte_bucket"], observed=True)
    out = g.agg(n=("log_iv_ratio", "size"), median_log_ratio=("log_iv_ratio", "median"),
                median_T_min=("T", lambda s: float(np.median(s) * va.MINUTES_PER_YEAR))).reset_index()
    out["implied_delta_min"] = out["median_T_min"] * (np.exp(2 * out["median_log_ratio"]) - 1)
    out["subset"] = subset
    return out
partB_tte = pd.concat([_tte_profile(a, "all"), _tte_profile(a[a["vega_ok"]], Z_LABEL)], ignore_index=True)
print("Implied delta (minutes) by TTE bucket: a real offset is the same in every bucket")
display(partB_tte.pivot_table(index=["subset", "underlying"], columns="tte_bucket", values="implied_delta_min", observed=True).round(2))
display(partB_tte.pivot_table(index=["subset", "underlying"], columns="tte_bucket", values="n", observed=True))

## Part C — ATM IV feature vs straddle-implied IV

Per state: the strike nearest the forward with two-sided call **and** put quotes; the volatility that reprices the straddle mid (with our $T$) is compared with `option_iv_atm_forward` from the state table.

In [ ]:
cp = q2.pivot_table(index=["state_id", "underlying", "strike"], columns="is_call",
                    values=["mid_usd", "option_forward_price", "T", "minutes_to_expiry"], aggfunc="first")
cp.columns = [f"{v}_{'c' if c else 'p'}" for v, c in cp.columns]
if not {"mid_usd_c", "mid_usd_p"} <= set(cp.columns):
    raise RuntimeError("Part C: no strike has both a call and a put quote; check option_type values above.")
cp = cp.dropna(subset=["mid_usd_c", "mid_usd_p"]).reset_index()
cp["F"] = cp["option_forward_price_c"]
cp["dist"] = (np.log(cp["strike"] / cp["F"])).abs()
atm = cp.loc[cp.groupby("state_id")["dist"].idxmin()].copy()
atm["straddle_iv"] = op.straddle_implied_vol(atm["mid_usd_c"] + atm["mid_usd_p"], atm["F"], atm["strike"], atm["T_c"])

feat = df[["state_id", "option_iv_atm_forward", "option_atm_quote_ok", "in_settlement_window"]].copy()
v = pd.to_numeric(feat["option_iv_atm_forward"], errors="coerce")
feat["iv_feature"] = v / 100.0 if np.nanmedian(v) > 3 else v
atm = atm.merge(feat[["state_id", "iv_feature"]], on="state_id", how="inner")
atm["log_ratio"] = np.log(atm["iv_feature"] / atm["straddle_iv"])
atm["tte_bucket"] = va.tte_bucket(atm["minutes_to_expiry_c"].to_numpy(float)).astype(str)
partC = (atm.groupby(["underlying", "tte_bucket"], observed=True)["log_ratio"]
            .agg(["size", "median", lambda s: float(np.nanmedian(np.abs(s)))]).reset_index())
partC.columns = ["underlying", "tte_bucket", "n", "median_log_ratio", "median_abs_log_ratio"]
print("log(IV feature / straddle-implied IV):")
display(partC.round(4))

## Part D — Where does the terminal / 30m sign flip come from?

For every origin with a quote-backed ATM IV, every realized and implied definition is computed (`vrp_audit.variance_definitions`):

* **flat**: Black-76 pricing variance $IV^2\,\Delta t$ (what a delta-hedged option's theta pays);
* **clock**: Module 12's clock-allocated implied variance;
* **_w**: settlement-average weighting $w_j^2$ applied to both sides.

`log_ratio_of_sums` $=\log(\sum RV/\sum IV^2\Delta t)$ is the aggregate a variance seller's P&L scales with; the median log ratio is reported beside it. The comparison is shown on each definition's own rows and on **common rows** (all definitions available), which removes sample-composition differences.

In [ ]:
DEFS, tables, tables_common = {}, [], []
for asset in ASSETS:
    g = df[df["underlying"] == asset].sort_values("state_time").reset_index(drop=True)
    panel = vt.build_panel(g)
    nuis = vh.fit_nuisance(g, panel, CLOCK_TRAIN)
    alloc = vt.clock_allocation(panel, nuis.clock_set().fitted[0])
    implied = vt.implied_variance(panel, alloc)
    d = va.variance_definitions(panel, implied)
    DEFS[asset] = d
    tables.append(va.decomposition_table(d, asset, common_rows=False))
    tables_common.append(va.decomposition_table(d, asset, common_rows=True))
partD = pd.concat(tables, ignore_index=True)
partD_common = pd.concat(tables_common, ignore_index=True)

show = ["underlying", "comparison", "n", "sessions", "median_log_ratio", "log_ratio_of_sums", "share_rv_above_imp",
        "session_median_min", "session_median_max"]
print("All TTE, each definition on its own rows:")
display(partD[partD.tte_bucket == "all"][show].round(3))
print("All TTE, common rows:")
display(partD_common[partD_common.tte_bucket == "all"][show].round(3))
print("By TTE bucket, common rows (log ratio of sums):")
display(partD_common[partD_common.tte_bucket != "all"]
        .pivot_table(index=["underlying", "tte_bucket"], columns="comparison", values="log_ratio_of_sums").round(3))

### Attribution of the flip

For each asset the steps below move one ingredient at a time from Module 12's terminal definition to its 30m definition. The ingredient whose step changes the sign is the cause.

In [ ]:
steps = [
    ("M12 terminal: exp_clock_w", "exp_clock_w (M12 terminal)"),
    ("drop clock allocation: exp_flat_w", "exp_flat_w"),
    ("drop settlement weighting: exp_flat", "exp_flat"),
    ("change horizon to 30m: 30m_flat", "30m_flat"),
    ("add clock allocation: 30m_clock (M12 intraday)", "30m_clock (M12 intraday)"),
]
attr = []
for asset in ASSETS:
    for view, tab in (("own rows", partD), ("common rows", partD_common)):
        t = tab[(tab.underlying == asset) & (tab.tte_bucket == "all")].set_index("comparison")
        row = {"underlying": asset, "rows": view}
        for label, comp in steps:
            row[label] = t.loc[comp, "median_log_ratio"] if comp in t.index else np.nan
        attr.append(row)
attr = pd.DataFrame(attr)
print("Median log(RV / implied) as one ingredient changes at a time:")
display(attr.round(3))

## Part E — Microstructure noise in 5-minute index returns

Per session: RV from 5-minute returns divided by RV from non-overlapping 30-minute returns over the same span. Under a martingale the ratio is about 1.

In [ ]:
partE = []
for asset in ASSETS:
    g = df[df["underlying"] == asset].sort_values("state_time").reset_index(drop=True)
    s = va.sampling_ratio(vt.build_panel(g), step=6)
    s.insert(0, "underlying", asset)
    partE.append(s)
partE = pd.concat(partE, ignore_index=True)
summaryE = partE.groupby("underlying").agg(sessions=("session", "size"),
                                           median_ratio=("ratio_5m_over_coarse", "median"),
                                           q25=("ratio_5m_over_coarse", lambda s: s.quantile(0.25)),
                                           q75=("ratio_5m_over_coarse", lambda s: s.quantile(0.75)),
                                           median_lag1_ac=("lag1_autocorr", "median"))
display(summaryE.round(3))

## Gate summary

In [ ]:
gates = []
for asset in ASSETS:
    rowB = partB[(partB.underlying == asset) & (partB.subset == Z_LABEL)]
    rowC = partC[partC.underlying == asset]
    cmed = float(np.nanmedian(atm.loc[atm.underlying == asset, "log_ratio"].abs())) if (atm.underlying == asset).any() else np.nan
    gates.append({
        "underlying": asset,
        "A_median_abs_err_hs": float(overall_A.get(asset, np.nan)),
        "A_pass": bool(overall_A.get(asset, np.inf) <= 0.25),
        "B_delta_minutes": float(rowB["delta_minutes"].iloc[0]) if len(rowB) and "delta_minutes" in rowB else np.nan,
        "B_pass": bool(len(rowB) and "delta_minutes" in rowB and abs(rowB["delta_minutes"].iloc[0]) <= 2.0),
        "C_median_abs_log_ratio": cmed,
        "C_tte_spread": float(rowC["median_log_ratio"].max() - rowC["median_log_ratio"].min()) if len(rowC) else np.nan,
        "C_pass": bool(cmed <= 0.02 and len(rowC) and (rowC["median_log_ratio"].abs() <= 0.03).all()),
        "E_median_ratio": float(summaryE.loc[asset, "median_ratio"]) if asset in summaryE.index else np.nan,
        "E_pass": bool(asset in summaryE.index and 0.8 <= summaryE.loc[asset, "median_ratio"] <= 1.25),
    })
gates = pd.DataFrame(gates)
display(gates.round(4))

partA.to_csv(OUT / "15_partA_repricing.csv", index=False)
partB.to_csv(OUT / "15_partB_time_basis.csv", index=False)
partB_tte.to_csv(OUT / "15_partB_by_tte.csv", index=False)
partC.to_csv(OUT / "15_partC_atm_feature.csv", index=False)
partD.to_csv(OUT / "15_partD_variance_own_rows.csv", index=False)
partD_common.to_csv(OUT / "15_partD_variance_common_rows.csv", index=False)
attr.to_csv(OUT / "15_partD_attribution.csv", index=False)
partE.to_csv(OUT / "15_partE_sampling.csv", index=False)
gates.to_csv(OUT / "15_gates.csv", index=False)
print("Wrote 15_*.csv to", OUT)

## Interpretation rules

* **A or B fails**: our time to expiry or price units disagree with the exchange. Fix the convention in `normalized_option_chain` / `option_pricing` before any P&L work; every option P&L would otherwise be mis-stated.
* **C fails**: `option_iv_atm_forward` is not the straddle's volatility. The P&L branch then uses straddle-implied IV (what the trade actually pays), and Module 12's M2a feature is re-examined.
* **D**: if the flip disappears when the clock allocation is removed, Module 12's intraday "options rich" reading was an allocation artifact; the flat $IV^2\Delta t$ comparison is the economically relevant one (018 §2.2).
* **E fails high**: 5-minute RV overstates diffusive variance against implied; P&L tests must use the hedge-frequency-matched realized variance.
* The unconditional premium itself (step 2) is measured only after A–C pass.